# Part 1 – Scraping emergency decrees

**Season:** January 1 to May 31, 2023
**Source:** gob.pe – PCM regulation search
**Output:** `datos/decretos_lluvias.csv` and `datos/decretos_por_departamento.csv`

## 1. Goal

Collect every supreme decree that declares or extends a state of emergency caused by heavy rainfall during the 2023 season, enrich each one with its full title, classify them, and count how many mention each department of Peru.

### Language conventions

Everything written by us is in English: prose, comments, docstrings, printed messages and Python identifiers.

Three things stay in Spanish on purpose:

1. **Column names and category values.** The assignment specifies them literally (`numero`, `fecha`, `titulo`, `tipo`, `motivo`, `enlace`, `departamento`, `declaratorias`, `prorrogas`, and the values `declara`, `prorroga`, `otro`, `peligro inminente`, `impacto de daños`). Parts 2 and 3 join on these names, so renaming them would break the other notebooks.
2. **File and folder names.** Same reason: the assignment lists `datos/decretos_lluvias.csv` and `datos/decretos_por_departamento.csv` by name.
3. **The scraped records themselves**: decree numbers, titles, publication dates and links. A decree title is the legal name of a document published by the Peruvian government. Translating it would falsify the data, not localize it.

Department names **are** written without accents in the output data (`Ancash`, not `Áncash`). Section 12 explains how the search still matches the accented spelling that gob.pe actually uses.

## 2. Imports

In [1]:
import os
import re
import time
import calendar
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 3. Configuration

Project constants: base URL, year, months, pauses, retry budget and User-Agent.

In [2]:
# Base URL of the gob.pe regulation search
BASE_URL = "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent"

# Season year
YEAR = 2023

# Months to scrape (1 = January, 2 = February, ..., 5 = May)
MONTHS = [1, 2, 3, 4, 5]

# Pause between Selenium pages, in seconds
SELENIUM_PAUSE = 2

# Pause between requests calls, in seconds
REQUEST_PAUSE = 1

# Retries per page when the connection fails, and timeout for each request
RETRIES = 4
REQUEST_TIMEOUT = 30

# User-Agent so gob.pe sees us as a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Departments of Peru, written without accents because that is how they must
# appear in the output data. Section 12 handles the accented spelling used in
# the decree titles.
DEPARTMENTS = [
    "Amazonas", "Ancash", "Apurimac", "Arequipa", "Ayacucho",
    "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huanuco",
    "Ica", "Junin", "La Libertad", "Lambayeque", "Lima", "Loreto",
    "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
    "San Martin", "Tacna", "Tumbes", "Ucayali"
]

## 4. robots.txt

Before scraping anything, we read the `robots.txt` file of gob.pe to understand what is allowed and what is not.

In [3]:
# Download robots.txt
response = requests.get("https://www.gob.pe/robots.txt", headers=HEADERS)
robots_txt = response.text
print(robots_txt[:2000])  # First 2000 characters

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



### Reading the robots.txt

**What does it forbid?** For every bot (`User-agent: *`), gob.pe blocks three paths:

- `Disallow: /admin/`, the site administration panel.
- `Disallow: /*?sheet=` and `Disallow: /*&sheet=`, any URL carrying the `sheet` parameter, which triggers table exports. This keeps bots from firing expensive exports.

There are also per-bot rules: `AhrefsBot`, a commercial SEO crawler, is blocked from **the entire site** with `Disallow: /`.

**Is the search page allowed?** Yes. `/busquedas` does not appear in any `Disallow` rule, neither in the general block nor in the per-bot ones. Our path is allowed.

**Why pause anyway?** Because the file itself tells us the pace gob.pe considers acceptable. It asks the AI crawlers explicitly:

- `GPTBot` gets `Crawl-delay: 5`, five seconds between requests.
- `OAI-SearchBot` gets `Crawl-delay: 2`, two seconds between requests.

There is no `Crawl-delay` under `User-agent: *`, so none is formally binding on us. But a missing rule is not permission to hammer the server: if a commercial crawler is asked for two to five seconds, that is the pace the site tolerates. So we use **two seconds between search pages** (`SELENIUM_PAUSE`) and **one second between decree pages** (`REQUEST_PAUSE`), for roughly fifty lightweight requests in total.

The pauses also protect us. Without them gob.pe starts dropping connections on timeout, and a failed request means a decree that silently disappears from the analysis.

## 5. Month by month scraping with Selenium

The gob.pe search page loads its results with JavaScript, so we need Selenium. We query one month at a time, which is what makes the per-month verification possible.

In [4]:
def month_date_range(month, year):
    """
    Return the first and last day of a month as DD-MM-YYYY strings.
    The last day is computed, not hardcoded, so February works on leap years.
    """
    last_day = calendar.monthrange(year, month)[1]
    return f"01-{month:02d}-{year}", f"{last_day}-{month:02d}-{year}"


def fetch_month_results(driver, month, year):
    """
    Open the search page for one month and extract every result on it.
    Return the list of results and the total the page itself reports.
    """
    start, end = month_date_range(month, year)
    url = f"{BASE_URL}&desde={start}&hasta={end}"

    driver.get(url)

    # Wait for the results to render, since the page builds them with JavaScript
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
    )

    # Read the result count the page declares. No try/except here on purpose:
    # if this breaks, the verification table would silently compare 0 against 0.
    total_element = driver.find_element(By.XPATH, "//span[contains(., 'Resultados')]")
    total = int(total_element.text.split()[0])

    # Every result is an <article>
    articles = driver.find_elements(By.CSS_SELECTOR, "article")

    results = []
    for article in articles:
        try:
            # The decree number is the text of the main link
            main_link = article.find_element(By.CSS_SELECTOR, "h4 a.link-principal")
            number = main_link.text.strip()
            decree_url = main_link.get_attribute("href")

            # The publication date sits in a span with the 'text-gray-900' class
            date_element = article.find_element(By.CSS_SELECTOR, "span[class*='text-gray-900']")
            date = date_element.text.strip()

            # The title is the paragraph whose id contains 'description'
            title_element = article.find_element(By.CSS_SELECTOR, "p[id*='description']")
            title = title_element.text.strip()

            results.append({
                "numero": number,
                "fecha": date,
                "titulo": title,
                "enlace": decree_url
            })
        except Exception:
            # Skip a result whose markup does not match the expected shape
            continue

    return results, total

### Running the scrape for every month

We process January through May 2023, pausing two seconds between pages.

In [5]:
# Configure the Chrome driver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Do not open a browser window
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

all_results = []
monthly_totals = []

for month in MONTHS:
    print(f"Processing month {month}...")
    month_results, total = fetch_month_results(driver, month, YEAR)

    # Keep the total the page reported, for the verification table
    monthly_totals.append({"mes": month, "resultados_totales": total})

    # Tag every result with its month
    for result in month_results:
        result["mes"] = month

    all_results.extend(month_results)
    print(f"  Extracted {len(month_results)} of {total} reported")

    # Pause between pages
    time.sleep(SELENIUM_PAUSE)

driver.quit()
print(f"\nTotal regulations scraped: {len(all_results)}")

Processing month 1...


  Extracted 11 of 11 reported


Processing month 2...


  Extracted 9 of 9 reported


Processing month 3...


...

Processing month 4...


  Extracted 11 of 11 reported


Processing month 5...


  Extracted 8 of 8 reported



Total regulations scraped: 53


## 6. Verification

We compare what we extracted against the total the page reports, month by month. The column names follow the assignment: `mes`, `resultados_totales`, `resultados_extraidos`.

In [6]:
# Build a DataFrame with every result
df = pd.DataFrame(all_results)

# Count how many we extracted per month
extracted_per_month = df.groupby("mes").size().reset_index(name="resultados_extraidos")

# Build the verification table
verification_table = pd.DataFrame(monthly_totals)
verification_table = verification_table.merge(extracted_per_month, on="mes", how="left")
verification_table["resultados_extraidos"] = verification_table["resultados_extraidos"].fillna(0).astype(int)
verification_table["coincide"] = verification_table["resultados_totales"] == verification_table["resultados_extraidos"]

print(verification_table)
print()

# A mismatch means the page paginated the results or changed its markup.
# Carrying on would produce incomplete counts, so we stop instead.
if verification_table["coincide"].all():
    print("All months match. Safe to continue.")
else:
    failed = verification_table.loc[~verification_table["coincide"], "mes"].tolist()
    raise RuntimeError(f"Months {failed} do not match: check pagination or the CSS selectors.")

   mes  resultados_totales  resultados_extraidos  coincide
0    1                  11                    11      True
1    2                   9                     9      True
2    3                  14                    14      True
3    4                  11                    11      True
4    5                   8                     8      True

All months match. Safe to continue.


## 7. Removing duplicates

A decree can show up in more than one search, for instance when it was published close to a month boundary. We deduplicate by link.

In [7]:
# Count the duplicates
duplicates = df["enlace"].duplicated().sum()
print(f"Duplicates found: {duplicates}")

# Drop them, keeping the first occurrence
df = df.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"Records after deduplication: {len(df)}")

Duplicates found: 0
Records after deduplication: 53


## 8. PCM filter

The search is supposed to filter by PCM, but regulations from other institutions still slip through (OECE resolutions, for example). We keep only the ones whose link contains `/institucion/pcm/` and report how many we dropped.

In [8]:
# Keep only PCM regulations
df_pcm = df[df["enlace"].str.contains("/institucion/pcm/", na=False)].copy()

removed_non_pcm = len(df) - len(df_pcm)
print(f"Non-PCM regulations removed: {removed_non_pcm}")
print(f"PCM regulations remaining: {len(df_pcm)}")

Non-PCM regulations removed: 4
PCM regulations remaining: 49


## 9. Full titles with requests and BeautifulSoup

Titles in the result list are truncated with an ellipsis, and what gets cut is usually the reason for the emergency. Each decree page is plain HTML, so no Selenium is needed here: `requests` plus BeautifulSoup is enough.

In [9]:
def fetch_full_title(url):
    """
    Fetch one decree page and return its full title.
    Retries with a growing wait: a lost page makes that decree vanish from
    every later step, and it does so without leaving a trace.
    """
    for attempt in range(1, RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # The full title lives in a div with the 'description' class
            description_div = soup.find("div", class_="description")
            if description_div is None:
                return None

            title = description_div.text.strip()
            # Strip the trailing boilerplate ("DS N° ... PDF ... Descargar")
            title = re.sub(r"\s*DS\s+N[°º].*$", "", title, flags=re.IGNORECASE | re.DOTALL)
            title = re.sub(r"\s+PDF\s+.*$", "", title, flags=re.IGNORECASE)
            title = re.sub(r"\s+Descargar\s*$", "", title)
            return title
        except requests.RequestException as error:
            print(f"  Attempt {attempt}/{RETRIES} failed for {url}: {error}")
            if attempt < RETRIES:
                time.sleep(REQUEST_PAUSE * attempt)
    return None


# Fetch the full title of every decree, pausing between requests
print("Fetching full titles...")
titles = []
for url in df_pcm["enlace"]:
    titles.append(fetch_full_title(url))
    time.sleep(REQUEST_PAUSE)

df_pcm["titulo_completo"] = titles

missing = df_pcm[df_pcm["titulo_completo"].isna()]
print(f"Titles fetched: {len(df_pcm) - len(missing)} of {len(df_pcm)}")

# Without the full title we cannot classify a decree or detect its departments.
# Continuing here would skew every count downstream, silently.
if len(missing) > 0:
    print("\nRegulations with no full title:")
    for _, row in missing.iterrows():
        print(f"  - {row['numero']}: {row['enlace']}")
    raise RuntimeError(
        f"{len(missing)} regulations have no full title. "
        "Re-run this cell before continuing."
    )

Fetching full titles...


Titles fetched: 49 of 49


## 10. Classification: es_lluvia, tipo, motivo

We build the classification columns from the lowercased full title. Both the column names and the category values are the ones the assignment specifies, so they stay in Spanish.

Note that we do **not** strip accents here. Removing them would turn `daños` into `danos`, and the `impacto de daños` test would stop matching. Accent stripping belongs only to the department search in section 12.

In [10]:
# Lowercased title, to keep the membership tests simple
df_pcm["titulo_minusculas"] = df_pcm["titulo_completo"].str.lower()

# es_lluvia: True when the title mentions rain or rainfall
df_pcm["es_lluvia"] = (
    df_pcm["titulo_minusculas"].str.contains("lluvia", na=False) |
    df_pcm["titulo_minusculas"].str.contains("precipitaciones", na=False)
)


def classify_type(title):
    """Map a lowercased title to prorroga, declara or otro."""
    if "prórroga" in title or "prorroga" in title:
        return "prorroga"
    if "declara" in title:
        return "declara"
    return "otro"


df_pcm["tipo"] = df_pcm["titulo_minusculas"].apply(classify_type)


def classify_reason(title):
    """Map a lowercased title to peligro inminente, impacto de daños or otro."""
    if "peligro inminente" in title:
        return "peligro inminente"
    if "impacto de daños" in title:
        return "impacto de daños"
    return "otro"


df_pcm["motivo"] = df_pcm["titulo_minusculas"].apply(classify_reason)

print(df_pcm["tipo"].value_counts().to_string())
print()
print(df_pcm["motivo"].value_counts().to_string())

# Review the rows that fell into 'otro'.
# The detail is listed for rainfall regulations only, since those are the ones
# that reach the final CSV: the rest are dropped in the next step.
for column in ["tipo", "motivo"]:
    others = df_pcm[df_pcm[column] == "otro"]
    rain_others = others[others["es_lluvia"]]
    print(f"\n=== {column} == 'otro': {len(others)} of {len(df_pcm)} PCM regulations")
    print(f"    of which rainfall related: {len(rain_others)}")
    for _, row in rain_others.iterrows():
        print(f"  - {row['numero']}: {row['titulo_completo']}")

tipo
declara     28
prorroga    21

motivo
otro                 28
impacto de daños     16
peligro inminente     5

=== tipo == 'otro': 0 of 49 PCM regulations
    of which rainfall related: 0

=== motivo == 'otro': 28 of 49 PCM regulations
    of which rainfall related: 2
  - Decreto Supremo N.° 043-2023-PCM: Decreto Supremo que declara el Estado de Emergencia Nacional, por desastre de gran magnitud, a consecuencia de intensas precipitaciones pluviales en los departamentos de Lambayeque, Piura y Tumbes
  - Decreto Supremo N.° 065-2023-PCM: Decreto Supremo que prorroga el Estado de Emergencia Nacional, por desastre de gran magnitud, a consecuencia de intensas precipitaciones pluviales en varios distritos de las provincias de los departamentos de Lambayeque, Piura y Tumbes


### Reviewing the rows classified as "otro"

**`tipo == "otro"`: 0 rows.** All 49 PCM regulations land in `declara` (28) or `prorroga` (21). Any regulation that declares or extends a state of emergency uses one of those two verbs in its title, so the classification is complete.

**`motivo == "otro"`: 28 rows, but only 2 of them are rainfall related.** The other 26 are dropped in the next step for not being about rain: emergencies from other causes (national road network, road corridors) and even two 2015 directives about El Niño drills that slipped into the search.

The 2 rainfall rows left in `"otro"` are:

- `Decreto Supremo N.° 043-2023-PCM`, which declares a **national** state of emergency *"por desastre de gran magnitud"* over Lambayeque, Piura and Tumbes.
- `Decreto Supremo N.° 065-2023-PCM`, the extension of that same national emergency, using the same wording.

**Are they correctly classified? Yes.** And this is a finding, not a bug: these two decrees use a **third reason** the assignment did not list, *"por desastre de gran magnitud"* (major disaster). They are the national state of emergency declared for Cyclone Yaku in March 2023, which hit the north of the country.

The distinction is substantive, not editorial:

- *peligro inminente* is declared **before** the damage, on a forecast;
- *impacto de daños* is declared **after** the damage, scoped to specific districts;
- *desastre de gran magnitud* is the SINAGERD category for events that exceed regional response capacity and escalate to the national level.

Forcing these two decrees into one of the two requested categories would be inventing data. We leave them as `"otro"` and document why, which is the honest answer.

This matters for Part 3: when computing the share of declarations by imminent hazard versus damage impact, these 2 rows belong to neither group and have to be handled separately, or the percentages will not add up to 100%.

## 11. Keeping only the rainfall regulations

From here on we work with the regulations whose title mentions rain or rainfall.

In [11]:
# Keep only the rainfall regulations
df_rain = df_pcm[df_pcm["es_lluvia"]].copy()

print(f"Rainfall regulations: {len(df_rain)}")
print("\nFirst 5 rows:")
df_rain[["numero", "fecha", "titulo_completo", "tipo", "motivo"]].head()

Rainfall regulations: 20

First 5 rows:


,numero,fecha,titulo_completo,tipo,motivo
8,Decreto Supremo N.° 008-2023-PCM,Publicado: 13 de enero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,peligro inminente
9,Decreto Supremo N.° 007-2023-PCM,Publicado: 13 de enero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
13,Decreto Supremo N.° 024-2023-PCM,Publicado: 17 de febrero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
18,Decreto Supremo N.° 019-2023-PCM,Publicado: 6 de febrero de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños
21,Decreto Supremo N.° 044-2023-PCM,Publicado: 29 de marzo de 2023,Decreto Supremo que declara el Estado de Emerg...,declara,impacto de daños


## 12. Department detection

We identify which departments each decree mentions. A decree can mention several.

Two problems have to be solved at once here. The output data must use department names **without** accents, but gob.pe writes them **with** accents (`Áncash`, `Huánuco`, `San Martín`). Searching the accent-free name against the raw title would find nothing: `Áncash` alone accounts for 7 of the 20 rainfall decrees, and it would silently drop to 0.

So we strip the accents from the **title** before searching, and keep the accent-free name from `DEPARTMENTS` as the value we store.

In [12]:
def strip_accents(text):
    """
    Remove accent marks so 'Áncash' compares equal to 'Ancash'.
    Only used for department matching: it also turns 'daños' into 'danos',
    which is why the classification in section 10 works on the raw title.
    """
    decomposed = unicodedata.normalize("NFD", text)
    return "".join(char for char in decomposed if unicodedata.category(char) != "Mn")


def find_departments(title):
    """
    Return the departments mentioned in a title, without repetitions.
    Matching uses word boundaries (\\b) so 'Ica' does not match inside
    'Huancavelica', and each department is appended once even when the title
    names it twice, as in 'provincia de Ica del departamento de Ica'.
    """
    plain_title = strip_accents(title)
    found = []
    for department in DEPARTMENTS:
        if re.search(r"\b" + re.escape(department) + r"\b", plain_title):
            found.append(department)
    return found


# Detect the departments of every decree
df_rain["departamentos"] = df_rain["titulo_completo"].apply(find_departments)

# declaratorias and prorrogas are disjoint categories: each decree counts once.
rows = []
for department in DEPARTMENTS:
    mentions = df_rain["departamentos"].apply(lambda found: department in found)
    rows.append({
        "departamento": department,
        "declaratorias": int((mentions & (df_rain["tipo"] == "declara")).sum()),
        "prorrogas": int((mentions & (df_rain["tipo"] == "prorroga")).sum()),
    })

df_departments = pd.DataFrame(rows)

print(df_departments.to_string(index=False))
print()

# No decree may be lost when split across the two columns
column_total = df_departments["declaratorias"].sum() + df_departments["prorrogas"].sum()
mention_total = df_rain["departamentos"].apply(len).sum()
print(f"Department-decree mentions: {mention_total}")
print(f"declaratorias + prorrogas:  {column_total}")
if column_total != mention_total:
    raise RuntimeError("Some decree has a 'tipo' other than declara or prorroga: check the classification.")

 departamento  declaratorias  prorrogas
     Amazonas              3          1
       Ancash              6          1
     Apurimac              1          0
     Arequipa              4          0
     Ayacucho              5          1
    Cajamarca              4          0
       Callao              1          0
        Cusco              2          0
 Huancavelica              3          0
      Huanuco              4          0
          Ica              3          0
        Junin              2          0
  La Libertad              5          0
   Lambayeque              3          1
         Lima              6          0
       Loreto              1          0
Madre de Dios              0          0
     Moquegua              2          0
        Pasco              2          0
        Piura              3          1
         Puno              1          0
   San Martin              4          0
        Tacna              2          0
       Tumbes              2          1


In [13]:
# Checks for the three ways department detection can go wrong,
# using real decrees from our season where possible.

# Risk 1: double counting. A decree naming Ica as a province AND as a department.
ica_cases = df_rain[df_rain["titulo_completo"].str.contains("del departamento de Ica", na=False)]
for _, row in ica_cases.iterrows():
    print(f"{row['numero']}")
    print(f"  Title: {row['titulo_completo']}")
    print(f"  Times 'Ica' appears as a word: {len(re.findall(r'\bIca\b', row['titulo_completo']))}")
    print(f"  Departments detected: {row['departamentos']}")
    print(f"  'Ica' listed exactly once? {row['departamentos'].count('Ica') == 1}")
    print()

# Risk 2: false positive. 'ica' lives inside 'Huancavelica'.
sample = "Decreto Supremo que declara el Estado de Emergencia en el departamento de Huancavelica"
print(f"Without \\b -> {'ica' in sample.lower()}  (false positive: would count Ica)")
print(f"With \\b    -> {find_departments(sample)}  (correct: Huancavelica only)")
print()

# Risk 3: accents. gob.pe writes the accented spelling, our data uses the plain one.
accented = ["Áncash", "Apurímac", "Huánuco", "Junín", "San Martín"]
for name in accented:
    matched = df_rain["titulo_completo"].apply(
        lambda title, target=name: target in title
    ).sum()
    plain = strip_accents(name)
    counted = int(df_departments.loc[df_departments["departamento"] == plain, "declaratorias"].iloc[0] +
                  df_departments.loc[df_departments["departamento"] == plain, "prorrogas"].iloc[0])
    print(f"  '{name}' in titles: {matched:2}  ->  counted as '{plain}': {counted:2}")

Decreto Supremo N.° 044-2023-PCM
  Title: Decreto Supremo que declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañón del departamento de Huánuco; y, en algunos distritos de las provincias de Nasca, Ica y Palpa del departamento de Ica, por impacto de daños a consecuencia de intensas precipitaciones pluviales
  Times 'Ica' appears as a word: 2
  Departments detected: ['Huanuco', 'Ica']
  'Ica' listed exactly once? True

Without \b -> True  (false positive: would count Ica)
With \b    -> ['Huancavelica']  (correct: Huancavelica only)

  'Áncash' in titles:  7  ->  counted as 'Ancash':  7
  'Apurímac' in titles:  1  ->  counted as 'Apurimac':  1
  'Huánuco' in titles:  4  ->  counted as 'Huanuco':  4
  'Junín' in titles:  2  ->  counted as 'Junin':  2
  'San Martín' in titles:  4  ->  counted as 'San Martin':  4


### How we avoid miscounting departments

There are three separate ways to get this wrong, and each needs its own defense.

**Risk 1, false positive: `"ica"` lives inside `"Huancavelica"`.**

Searching the bare substring in a lowercased title makes `"ica" in "huancavelica"` return `True`, so a decree that only mentions Huancavelica would also count for Ica. The defense is to match on word boundaries with `\b`, which requires the match to start and end at a word edge:

```python
re.search(r"\b" + re.escape(department) + r"\b", plain_title)
```

The previous cell proves it: on `"...en el departamento de Huancavelica"` the bare substring test returns `True` for Ica, while ours returns `['Huancavelica']` only.

**Risk 2, double counting: "Ica" is also a province.**

The real case in our season is **`Decreto Supremo N.° 044-2023-PCM`**:

> *"...declara el Estado de Emergencia en algunos distritos de las provincias de Leoncio Prado y Marañón del departamento de Huánuco; y, en algunos distritos de las provincias de Nasca, **Ica** y Palpa del **departamento de Ica**, por impacto de daños a consecuencia de intensas precipitaciones pluviales"*

The word `Ica` appears **twice**, once as a province and once as a department. Counting occurrences would give Ica 2 for a single decree.

That does not happen, because `find_departments` asks about **presence, not quantity**: `re.search` returns the first match and `found.append(department)` runs once per department, inside a loop over the 25 names. The returned list cannot contain repetitions. The previous cell verifies it on this exact decree: 2 occurrences of the word, `Ica` listed once.

**Risk 3, accents: our data drops them, gob.pe does not.**

The output uses `Ancash`, `Apurimac`, `Huanuco`, `Junin` and `San Martin` without accents. But gob.pe publishes the accented spelling, so searching the plain name against the raw title matches nothing at all. This is the dangerous one, because nothing crashes: the affected departments would simply report 0.

Measured before the fix, on the 20 rainfall decrees of our season:

| Department | Accented, in titles | Plain, in titles |
|---|---|---|
| Áncash | 7 | 0 |
| Huánuco | 4 | 0 |
| San Martín | 4 | 0 |
| Junín | 2 | 0 |
| Apurímac | 1 | 0 |

That is 18 department-decree mentions that would have vanished without a single error message. The fix is `strip_accents`, applied to the **title** before searching, while the stored value is the accent-free name from `DEPARTMENTS`.

Note that accent stripping stays scoped to this step. `unicodedata` normalization also turns `ñ` into `n`, so applying it in section 10 would turn `daños` into `danos` and break the `impacto de daños` test. Same technique, wrong place.

**A limitation worth stating.** We only detect departments **named explicitly** in the title. A decree mentioning just *"la provincia de Nasca"*, without adding *"del departamento de Ica"*, would not be counted, because we do not resolve provinces or districts against a ubigeo registry. The count is therefore a floor, not an exact total.

### Criterion for `declaratorias` and `prorrogas`

The assignment names the columns `departamento`, `declaratorias` and `prorrogas` but never says whether they overlap. We treat them as **disjoint categories**:

- `declaratorias` counts decrees with `tipo == "declara"`
- `prorrogas` counts decrees with `tipo == "prorroga"`

That way the two columns can be summed without double counting, and the sum is the total number of decrees mentioning the department. Since `tipo == "otro"` has 0 rows in our season, nothing is lost in the split: the previous cell checks it by comparing the 78 department-decree mentions against the sum of both columns.

The alternative was to make `declaratorias` the total and `prorrogas` a subset of it, but then one column would be contained in the other with no name saying so, and anyone reading the CSV would sum them wrong.

## 13. Saving the files

We write both CSVs with `utf-8-sig` encoding so the remaining Spanish text in the decree titles renders correctly if the files are opened in Excel.

In [14]:
# Create the output folder if it does not exist
os.makedirs("datos", exist_ok=True)

# File 1: decretos_lluvias.csv
df_rain_output = df_rain[["numero", "fecha", "titulo_completo", "tipo", "motivo", "enlace"]].copy()
df_rain_output.columns = ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
df_rain_output.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
print(f"Saved datos/decretos_lluvias.csv ({len(df_rain_output)} rows)")

# File 2: decretos_por_departamento.csv
df_departments.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
print(f"Saved datos/decretos_por_departamento.csv ({len(df_departments)} rows)")

print("\nPart 1 completed.")

Saved datos/decretos_lluvias.csv (20 rows)
Saved datos/decretos_por_departamento.csv (25 rows)

Part 1 completed.


## 14. Limitations of this extraction

What this notebook does **not** guarantee, stated plainly so the output is not read as more than it is:

1. **No pagination.** We read the `<article>` elements on the first page of each month. No month of the 2023 season exceeds 14 results, so everything fits, and the verification table in section 6 **raises** if what we extracted does not match the total the page declares. In other words, if a month ever paginated, the notebook stops instead of quietly handing over incomplete data. A season with more regulations per month would need the following pages too.

2. **We only read the title, never the body of the decree.** The exact districts and provinces live in the PDF annexes. A department affected but not named in the title is not counted.

3. **The count is per decree, not per affected territory.** A decree covering 40 districts of Piura weighs the same as one covering a single district. Measuring magnitude would require counting districts, which is one of the improvements the assignment suggests as extra credit.

4. **The search depends on the term `"estado de emergencia precipitaciones"`.** A rainfall decree phrased with different wording might never appear in the results.

5. **The CSS selectors match the current gob.pe markup.** If the site changes its layout, the extraction stops finding the data. The verification table is what would surface that.